# NB09 — BacDive Geographic Niche Breadth PGLS

**Type:** Exploratory — alternative niche-breadth metric, independent data source  
**Status:** PENDING EXECUTION  

Derives genus-level geographic niche breadth from BacDive isolation metadata
(`kescience.bacdive.isolation`). Counts distinct countries (and distinct BacDive
geographic categories as a secondary axis) per genus after linking to GTDB taxonomy.
Tests whether BacDive-derived cosmopolitanism is negatively associated with
per-Mb metal-gene density from the comprehensive Tier 1+2 gene list.

## Pre-specifications

- Niche axis: number of distinct ISO countries of isolation per genus (primary);  
  number of distinct geographic categories per genus (secondary)
- Standardised niche breadth: B_std = (n_countries − 1) / (N_countries_max − 1),  
  clamped [0, 1]. N_countries_max set to 95th percentile across genera (avoids  
  leverage from highly sampled clades).
- Minimum coverage: ≥5 BacDive isolates per genus (any country)
- Predictor: `ko_per_mb_primary` from `data/01_genus_ko_density_spark.csv`  
  (Tier 1+2, 140 KOs, kescience_mgnify via Spark)
- Tree: `data/gtdb_bac_genus_pruned.tree`
- Expected direction: β < 0 (cosmopolitan genera → lower metal-gene density)
- Run once; no iterative tuning

## Rationale for expected direction

The primary PGLS (P1) shows that higher metal-gene KO density is associated
with **narrower** niche breadth (β < 0, p < 0.05): specialist genera invest more
genomic space in metal homeostasis. BacDive geographic spread (n_countries of
isolation) is a culture-based proxy for niche breadth — cosmopolitan genera
isolated across many countries are generalists. Expected: genera isolated across
many countries (high B_std) have lower KO/Mb → β < 0 when regressing B_std on
KO density.

## Expected outputs

- `data/bacdive_genus_country_counts.csv`
- `data/bacdive_niche_pgls_input.csv`
- `data/bacdive_niche_pgls_comprehensive.csv`
- `figures/09_bacdive_niche_vs_ko_density.png`
- INTERPRETATION_TABLE.md §7 updated

In [1]:
import sys
from pathlib import Path

_project_root = Path().resolve().parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scripts.pgls_utils import run_pgls, pgls_results_table
from scripts.berdl_utils import get_spark_session

DATA = Path('../data')
FIGS = Path('../figures')
FIGS.mkdir(exist_ok=True)
TREE_BAC = DATA / 'gtdb_bac_genus_pruned.tree'

pd.set_option('display.max_rows', 60)
print('Imports OK')


## Block 1 — Spark session + BacDive schema discovery

Describes the `kescience.bacdive` tables before querying — column names and exact
table availability are unknown and must be confirmed at runtime.


In [2]:
_spark = None
try:
    _spark = get_spark_session()
    print('Spark OK')
except BaseException as _e:
    print(f'Spark failed: {_e}')
    raise

# List all tables in kescience.bacdive
print('\n=== Tables in kescience.bacdive ===')
try:
    _tbls = _spark.sql('SHOW TABLES IN kescience.bacdive').toPandas()
    print(_tbls[['tableName', 'isTemporary']].to_string(index=False))
except Exception as _e:
    print(f'  SHOW TABLES failed: {_e}')
    # Try kescience namespace directly
    try:
        _tbls2 = _spark.sql('SHOW TABLES IN kescience LIKE "*bacdive*"').toPandas()
        print('kescience namespace:')
        print(_tbls2.to_string(index=False))
    except Exception as _e2:
        print(f'  kescience namespace also failed: {_e2}')

# Describe isolation table
def _describe_table(full_name):
    try:
        rows = _spark.sql(f'DESCRIBE TABLE {full_name}').toPandas()
        cols = [r for r in rows['col_name'].tolist() if not r.startswith('#') and r.strip()]
        print(f'\n{full_name} ({len(cols)} cols):')
        for c in cols:
            print(f'  {c}')
        return cols
    except Exception as e:
        print(f'  {full_name}: FAILED — {str(e)[:100]}')
        return []

_iso_cols   = _describe_table('kescience.bacdive.isolation')
_strain_cols = _describe_table('kescience.bacdive.strain')  # taxonomy linkage candidate
_tax_cols    = _describe_table('kescience.bacdive.taxonomy')  # alternative


## Block 2 — Locate taxonomy linkage table and key columns

BacDive isolates link to GTDB taxonomy via strain records. We need a join path
from `isolation` → strain/taxonomy → genus. Check all candidate tables.


In [5]:
# =============================================================================
# BacDive genus source mapping (isolation source categories)
# =============================================================================

# 1. Get column names for each table
try:
    iso_df_schema = _spark.sql("DESCRIBE kescience.bacdive.isolation").toPandas()
    _iso_cols = [r for r in iso_df_schema['col_name'].tolist() if not r.startswith('#') and r.strip()]
except Exception as e:
    print(f"Could not describe isolation table: {e}")
    _iso_cols = []

try:
    strain_df_schema = _spark.sql("DESCRIBE kescience.bacdive.strain").toPandas()
    _strain_cols = [r for r in strain_df_schema['col_name'].tolist() if not r.startswith('#') and r.strip()]
except Exception as e:
    print(f"Could not describe strain table: {e}")
    _strain_cols = []

print("Isolation columns:", _iso_cols[:10])
print("Strain columns:", _strain_cols[:10])

# 2. Define column mapping function
def _get_col(col_list, *candidates):
    lut = {c.lower().strip(): c for c in col_list if c.strip()}
    for cand in candidates:
        if cand.lower() in lut:
            return lut[cand.lower()]
    return None

# Map isolation table columns
iso_id   = _get_col(_iso_cols, 'bacdive_id', 'id', 'strain_id', 'record_id', 'bacdive_record_id')
iso_ctry = _get_col(_iso_cols, 'country', 'isolation_country', 'country_of_isolation',
                    'country_name', 'geo_country', 'origin_country', 'location_country')
iso_geo  = _get_col(_iso_cols, 'geographic_location', 'geographic_category', 'location',
                    'continent', 'region', 'environment_category', 'isolation_source_category')

print(f'\nIsolation mapping: id={iso_id}, country={iso_ctry}, geo={iso_geo}')

# Map strain table columns
tax_id      = _get_col(_strain_cols, 'bacdive_id', 'id', 'strain_id', 'record_id', 'bacdive_record_id')
tax_species = _get_col(_strain_cols, 'species_name', 'species', 'name', 'scientific_name')

print(f'Taxonomy mapping: id={tax_id}, species={tax_species}')

if not all([iso_id, iso_ctry, iso_geo, tax_id, tax_species]):
    raise ValueError("Missing required column mappings. Check table schemas.")

# 4. Load the data (limit to relevant columns to reduce memory)
iso_sql = f"""
    SELECT
        CAST(`{iso_id}` AS STRING) AS bacdive_id,
        `{iso_ctry}` AS country,
        `{iso_geo}` AS geo_location
    FROM kescience.bacdive.isolation
    WHERE `{iso_ctry}` IS NOT NULL OR `{iso_geo}` IS NOT NULL
"""
iso_df = _spark.sql(iso_sql).toPandas()
# Ensure string type (in case Spark returns something else)
iso_df['bacdive_id'] = iso_df['bacdive_id'].astype(str)
print(f"Loaded {len(iso_df):,} isolation records")

# Strain SQL – extract genus from species_name (first token before space)
strain_sql = f"""
    SELECT
        CAST(`{tax_id}` AS STRING) AS bacdive_id,
        LOWER(TRIM(SPLIT(`{tax_species}`, ' ')[0])) AS genus_lower
    FROM kescience.bacdive.strain
    WHERE `{tax_species}` IS NOT NULL
      AND TRIM(`{tax_species}`) != ''
      AND LENGTH(TRIM(SPLIT(`{tax_species}`, ' ')[0])) > 1
"""
strain_df = _spark.sql(strain_sql).toPandas()
# Ensure string type for merge key
strain_df['bacdive_id'] = strain_df['bacdive_id'].astype(str)
print(f"Loaded {len(strain_df):,} strain records with genus extracted from species")

# 5. Join isolation with genus
merged = iso_df.merge(strain_df, on='bacdive_id', how='inner')
print(f"Joined records: {len(merged):,}")

# 6. Classify isolation source into broad categories
def _classify_source(country, geo):
    """
    Heuristic classification based on country and geographic_location strings.
    Returns one of: 'soil', 'plant', 'animal', 'water', 'other'
    """
    text = f"{country} {geo}".lower()
    if any(k in text for k in ('soil', 'terrestrial', 'sediment', 'ground')):
        return 'soil'
    if any(k in text for k in ('plant', 'rhizosphere', 'phyllosphere', 'root', 'leaf')):
        return 'plant'
    if any(k in text for k in ('animal', 'gut', 'rumen', 'skin', 'lung', 'feces', 'faecal', 'fecal')):
        return 'animal'
    if any(k in text for k in ('water', 'marine', 'ocean', 'freshwater', 'lake', 'river', 'pond', 'aquatic')):
        return 'water'
    return 'other'

merged['source_cat'] = merged.apply(
    lambda row: _classify_source(row['country'], row['geo_location']),
    axis=1
)

# 7. Aggregate per genus: get the most common source category (mode)
genus_source = (merged.groupby('genus_lower')['source_cat']
                .agg(lambda x: x.value_counts().index[0])
                .reset_index()
                .rename(columns={'source_cat': 'bacdive_source'}))

# Also get the count of records per genus for quality control
genus_counts = merged.groupby('genus_lower').size().reset_index(name='n_bacdive_records')
genus_source = genus_source.merge(genus_counts, on='genus_lower', how='left')

print(f"Generated source mapping for {len(genus_source):,} genera")
print(genus_source['bacdive_source'].value_counts())

# 8. Save to CSV
genus_source.to_csv(DATA / 'bacdive_genus_source.csv', index=False)
print("Saved: data/bacdive_genus_source.csv")

## Block 3 — Sample BacDive isolation table to confirm structure

Print 10 rows to verify country and geography column content before building
the full aggregation query.


In [7]:
# Preview isolation table
try:
    _preview = _spark.sql('SELECT * FROM kescience.bacdive.isolation LIMIT 10').toPandas()
    print('Isolation table preview:')
    print(_preview.to_string(max_cols=20))
except Exception as _e:
    print(f'Preview failed: {_e}')

# Preview taxonomy / strain table
# Define the taxonomy table name (used in the mapping cell)
tax_table = 'kescience.bacdive.strain'

# Preview isolation table
try:
    _preview = _spark.sql('SELECT * FROM kescience.bacdive.isolation LIMIT 10').toPandas()
    print('Isolation table preview:')
    print(_preview.to_string(max_cols=20))
except Exception as _e:
    print(f'Preview failed: {_e}')

# Preview taxonomy / strain table
try:
    _preview_tax = _spark.sql(f'SELECT * FROM {tax_table} LIMIT 10').toPandas()
    print(f'\n{tax_table} preview:')
    print(_preview_tax.to_string(max_cols=20))
except Exception as _e:
    print(f'Taxonomy preview failed: {_e}')
    

## Block 4 — Build country counts per genus

Joins isolation → taxonomy to get genus per isolate, then aggregates distinct
countries and geographic categories per genus. Falls back gracefully when
country column is null or when geographic column is missing.


In [10]:
# =============================================================================
# BacDive genus × country aggregation (using species_name to extract genus)
# =============================================================================

# Known table and column names
iso_table = 'kescience.bacdive.isolation'
tax_table = 'kescience.bacdive.strain'

iso_id   = 'bacdive_id'
iso_ctry = 'country'
iso_geo  = 'geographic_location'
tax_id   = 'bacdive_id'

# Build genus extraction from species_name (first word)
genus_expr = "LOWER(TRIM(SPLIT(t.species_name, ' ')[0]))"

# Build aggregation query dynamically
_ctry_agg = (
    f'COUNT(DISTINCT NULLIF(TRIM(CAST(i.`{iso_ctry}` AS STRING)), \'\')) AS n_countries,'
    if iso_ctry else
    'CAST(NULL AS BIGINT) AS n_countries,'
)
_geo_agg = (
    f'COUNT(DISTINCT NULLIF(TRIM(CAST(i.`{iso_geo}` AS STRING)), \'\')) AS n_geo_cats,'
    if iso_geo else
    'CAST(NULL AS BIGINT) AS n_geo_cats,'
)

agg_sql = f"""
SELECT
    {genus_expr} AS genus_lower,
    COUNT(*) AS n_isolates,
    {_ctry_agg}
    {_geo_agg}
    COUNT(DISTINCT i.`{iso_id}`) AS n_records
FROM {iso_table} i
JOIN {tax_table} t ON CAST(i.`{iso_id}` AS STRING) = CAST(t.`{tax_id}` AS STRING)
WHERE t.species_name IS NOT NULL
  AND TRIM(t.species_name) != ''
  AND LENGTH({genus_expr}) > 1
GROUP BY {genus_expr}
HAVING genus_lower IS NOT NULL AND genus_lower != ''
"""

print('Running BacDive genus × country aggregation using species_name...')
genus_ctry = _spark.sql(agg_sql).toPandas()

# Clean results
genus_ctry = genus_ctry[genus_ctry['genus_lower'].str.len() > 1].copy()
print(f'Result: {len(genus_ctry):,} genera')
print(f'n_isolates  : {genus_ctry["n_isolates"].sum():,} total records')
if 'n_countries' in genus_ctry.columns and genus_ctry['n_countries'].notna().any():
    print(f'n_countries : {genus_ctry["n_countries"].describe()}')
if 'n_geo_cats' in genus_ctry.columns and genus_ctry['n_geo_cats'].notna().any():
    print(f'n_geo_cats  : {genus_ctry["n_geo_cats"].describe()}')

# Save output
genus_ctry.to_csv(DATA / 'bacdive_genus_country_counts.csv', index=False)
print("Saved: data/bacdive_genus_country_counts.csv")

## Block 5 — Choose niche axis + compute standardised niche breadth

Primary axis: n_countries (ISO country count per genus).  
Fallback: n_geo_cats if country column is sparse (<10% non-null genera) or absent.  

Formula: B_std = (n − 1) / (N_max − 1), where N_max = 95th percentile of n across
genera with ≥5 isolates. B_std is clamped [0, 1].  
This differs from the Levins formula (which requires detection probabilities per
habitat) because BacDive provides a raw count metric — standardisation uses the
observed range rather than Σ p_i².


In [11]:
# Select niche axis
_has_ctry = (
    'n_countries' in genus_ctry.columns
    and genus_ctry['n_countries'].notna().sum() >= 0.10 * len(genus_ctry)
)
_has_geo = (
    'n_geo_cats' in genus_ctry.columns
    and genus_ctry['n_geo_cats'].notna().sum() >= 0.10 * len(genus_ctry)
)

if _has_ctry and genus_ctry['n_countries'].notna().sum() >= 0.5 * len(genus_ctry):
    _NICHE_COL   = 'n_countries'
    _NICHE_LABEL = 'n_countries (distinct ISO countries of isolation)'
elif _has_geo:
    _NICHE_COL   = 'n_geo_cats'
    _NICHE_LABEL = 'n_geo_cats (distinct BacDive geographic categories)'
else:
    # If neither is informative, use total n_isolates as a crude proxy
    _NICHE_COL   = 'n_isolates'
    _NICHE_LABEL = 'n_isolates (WARNING: using isolate count as proxy — country/geo columns empty)'
    print('WARNING: country and geographic category columns appear empty. '
          'Using n_isolates as crude proxy. Inspect table preview in Block 3.')

print(f'Selected niche axis: {_NICHE_LABEL}')
print(f'Non-null in selected column: {genus_ctry[_NICHE_COL].notna().sum():,} of {len(genus_ctry):,}')

# Apply ≥5 isolate minimum
genus_ctry_filt = genus_ctry[genus_ctry['n_isolates'] >= 5].copy()
print(f'\nGenera with ≥5 BacDive isolates: {len(genus_ctry_filt):,}')
print(f'  (dropped {len(genus_ctry) - len(genus_ctry_filt):,} with <5 isolates)')

genus_ctry_filt = genus_ctry_filt.dropna(subset=[_NICHE_COL])
print(f'After dropping null {_NICHE_COL}: {len(genus_ctry_filt):,}')

# Compute standardised breadth
_n_vals = genus_ctry_filt[_NICHE_COL].astype(float)
_N_max  = float(_n_vals.quantile(0.95))  # 95th percentile as ceiling
print(f'\nN_max (95th pctile of {_NICHE_COL}): {_N_max:.1f}')
print(f'Range of {_NICHE_COL}: {_n_vals.min():.0f} – {_n_vals.max():.0f}')

if _N_max <= 1:
    print('WARNING: N_max ≤ 1 — standardisation undefined. Using log1p-scaled values.')
    genus_ctry_filt['bacdive_B_std'] = np.log1p(_n_vals)
else:
    genus_ctry_filt['bacdive_B_std'] = ((_n_vals - 1) / (_N_max - 1)).clip(0, 1)

print(f'\nbacdive_B_std summary:')
print(genus_ctry_filt['bacdive_B_std'].describe())

genus_ctry_filt.to_csv(DATA / 'bacdive_genus_country_counts.csv', index=False)
print('\nSaved: data/bacdive_genus_country_counts.csv')


## Block 6 — Merge with MGnify KO density + phylum + PGLS


In [12]:
ko_df  = pd.read_csv(DATA / '01_genus_ko_density_spark.csv',
                      usecols=['genus_lower', 'ko_per_mb_primary'])
bac_tax = pd.read_csv(DATA / '01_pgls_input_bacteria.csv',
                       usecols=['genus_lower', 'phylum', 'kingdom'])

bdive_merged = (genus_ctry_filt[['genus_lower', 'bacdive_B_std', 'n_isolates',
                                   _NICHE_COL]]
                .merge(ko_df,   on='genus_lower', how='inner')
                .merge(bac_tax, on='genus_lower', how='inner')
                .dropna(subset=['bacdive_B_std', 'ko_per_mb_primary'])
                .copy())

print(f'BacDive PGLS input: {len(bdive_merged)} genera')
print(f'  B_std range  : {bdive_merged["bacdive_B_std"].min():.4f} – '
      f'{bdive_merged["bacdive_B_std"].max():.4f}')
print(f'  n_isolates   : {bdive_merged["n_isolates"].sum():,} total isolate records')
print(f'  KO/Mb range  : {bdive_merged["ko_per_mb_primary"].min():.2f} – '
      f'{bdive_merged["ko_per_mb_primary"].max():.2f}')

print('\nTop genera by B_std:')
print(bdive_merged.nlargest(10, 'bacdive_B_std')[
    ['genus_lower', 'bacdive_B_std', _NICHE_COL, 'ko_per_mb_primary']].to_string(index=False))

# Z-score predictor
bdive_merged['predictor_z'] = (
    (bdive_merged['ko_per_mb_primary'] - bdive_merged['ko_per_mb_primary'].mean())
    / bdive_merged['ko_per_mb_primary'].std()
)

print('\nRunning PGLS: bacdive_B_std ~ ko_per_mb_primary_z ...')
res_bdive = run_pgls(
    df=bdive_merged,
    tree_path=TREE_BAC,
    response='bacdive_B_std',
    predictors=['predictor_z'],
    taxon_col='genus_lower',
    label='BacDive_geo_niche_breadth',
    min_n=50,
)

print(f'\nBacDive PGLS result:')
print(f'  n   = {res_bdive.get("n")}')
print(f'  β   = {res_bdive.get("beta", float("nan")):.5f}')
print(f'  SE  = {res_bdive.get("SE",   float("nan")):.5f}')
print(f'  t   = {res_bdive.get("t_stat", float("nan")):.3f}')
print(f'  p   = {res_bdive.get("p_value", float("nan")):.3e}')
print(f'  λ   = {res_bdive.get("lambda_est", float("nan")):.3f}')
print(f'  r²  = {res_bdive.get("r2", float("nan")):.4f}')
print(f'  ΔAIC= {res_bdive.get("delta_aic_vs_null", float("nan")):.2f}')

_b = res_bdive.get('beta', 0)
_p = res_bdive.get('p_value', 1)
if _b < 0 and _p < 0.05:
    _direction = 'CONSISTENT with P1 — significant negative (β < 0, p < 0.05)'
elif _b < 0:
    _direction = 'Directionally consistent with P1 (β < 0), non-significant'
else:
    _direction = 'OPPOSITE to P1 (β > 0)'
print(f'\n  Pre-specified direction: β < 0')
print(f'  Observed: {_direction}')
print(f'  Niche axis: {_NICHE_LABEL}')

bdive_merged.to_csv(DATA / 'bacdive_niche_pgls_input.csv', index=False)
pd.DataFrame([res_bdive]).to_csv(DATA / 'bacdive_niche_pgls_comprehensive.csv', index=False)
print('\nSaved: data/bacdive_niche_pgls_input.csv')
print('Saved: data/bacdive_niche_pgls_comprehensive.csv')


## Block 7 — Secondary: n_geo_cats PGLS (if country is primary and geo available)

Runs a parallel PGLS using geographic category count as the niche axis. This
provides a check that the result is not sensitive to the exact niche metric.


In [13]:
res_bdive_geo = None

if _NICHE_COL == 'n_countries' and _has_geo:
    print('Running secondary PGLS: n_geo_cats axis')
    _geo_filt = genus_ctry_filt.copy()
    _geo_filt = _geo_filt.dropna(subset=['n_geo_cats'])
    _N_geo_max = float(_geo_filt['n_geo_cats'].quantile(0.95))
    if _N_geo_max > 1:
        _geo_filt['geo_B_std'] = ((_geo_filt['n_geo_cats'] - 1) / (_N_geo_max - 1)).clip(0, 1)
    else:
        _geo_filt['geo_B_std'] = np.log1p(_geo_filt['n_geo_cats'])

    _geo_merged = (_geo_filt[['genus_lower', 'geo_B_std', 'n_isolates']]
                   .merge(ko_df,   on='genus_lower', how='inner')
                   .merge(bac_tax, on='genus_lower', how='inner')
                   .dropna(subset=['geo_B_std', 'ko_per_mb_primary'])
                   .copy())
    _geo_merged['predictor_z'] = (
        (_geo_merged['ko_per_mb_primary'] - _geo_merged['ko_per_mb_primary'].mean())
        / _geo_merged['ko_per_mb_primary'].std()
    )

    if len(_geo_merged.dropna(subset=['predictor_z', 'geo_B_std'])) >= 50:
        res_bdive_geo = run_pgls(
            df=_geo_merged,
            tree_path=TREE_BAC,
            response='geo_B_std',
            predictors=['predictor_z'],
            taxon_col='genus_lower',
            label='BacDive_geo_cat_niche',
            min_n=50,
        )
        print(f'  geo_cats PGLS: n={res_bdive_geo.get("n")}, '
              f'β={res_bdive_geo.get("beta", float("nan")):.5f}, '
              f'p={res_bdive_geo.get("p_value", float("nan")):.3e}')
        pd.DataFrame([res_bdive_geo]).to_csv(
            DATA / 'bacdive_geocat_pgls.csv', index=False)
        print('  Saved: data/bacdive_geocat_pgls.csv')
    else:
        print(f'  geo_cats: n={len(_geo_merged)} < 50 — skipped')
else:
    print('Secondary geo_cats PGLS skipped '
          '(primary is not n_countries, or n_geo_cats column absent/sparse)')


## Block 8 — Figure


In [14]:
_n_panels = 2 + (1 if res_bdive_geo is not None else 0)
fig, axes = plt.subplots(1, _n_panels, figsize=(5 * _n_panels, 4))

# Distribution of B_std
axes[0].hist(bdive_merged['bacdive_B_std'], bins=40, color='#d62728', edgecolor='white')
axes[0].set_xlabel('BacDive geographic B_std')
axes[0].set_title(
    f'BacDive niche breadth\n({_NICHE_LABEL[:40]}, n={len(bdive_merged):,} genera)')

# Scatter
axes[1].scatter(bdive_merged['predictor_z'], bdive_merged['bacdive_B_std'],
                alpha=0.25, s=8, color='#d62728')
_xr = np.linspace(bdive_merged['predictor_z'].min(),
                   bdive_merged['predictor_z'].max(), 100)
if res_bdive.get('converged', False):
    _bval = res_bdive['beta']
    _ic   = bdive_merged['bacdive_B_std'].mean() - _bval * bdive_merged['predictor_z'].mean()
    axes[1].plot(_xr, _ic + _bval * _xr, 'r-', lw=2,
                  label=f"PGLS β={_bval:+.3f} (p={res_bdive['p_value']:.3g})")
axes[1].set_xlabel('KO density z (primary/Mb)')
axes[1].set_ylabel('BacDive geographic B_std')
axes[1].set_title(
    f'BacDive niche vs metal-gene density\n'
    f'(n={res_bdive.get("n","?")}, λ={res_bdive.get("lambda_est", float("nan")):.3f})')
axes[1].legend(fontsize=9)
axes[1].axhline(0, color='gray', lw=0.5, ls='--')

if res_bdive_geo is not None:
    axes[2].scatter(_geo_merged['predictor_z'], _geo_merged['geo_B_std'],
                    alpha=0.25, s=8, color='#ff7f0e')
    if res_bdive_geo.get('converged', False):
        _bg  = res_bdive_geo['beta']
        _icg = _geo_merged['geo_B_std'].mean() - _bg * _geo_merged['predictor_z'].mean()
        axes[2].plot(_xr, _icg + _bg * _xr, 'r-', lw=2,
                      label=f"β={_bg:+.3f} (p={res_bdive_geo['p_value']:.3g})")
    axes[2].set_xlabel('KO density z')
    axes[2].set_ylabel('geo_cat B_std')
    axes[2].set_title(f'Secondary: geo-category axis\n(n={res_bdive_geo.get("n","?")})')
    axes[2].legend(fontsize=9)

plt.tight_layout()
fig.savefig(FIGS / '09_bacdive_niche_vs_ko_density.png', dpi=150)
plt.show()
print('Saved: figures/09_bacdive_niche_vs_ko_density.png')


## Block 9 — Reporting summary

Copy values into INTERPRETATION_TABLE.md §7.


In [15]:
print('=== NB09 REPORTING SUMMARY ===')
print(f'Niche metric  : BacDive geographic B_std ({_NICHE_LABEL})')
print(f'Predictor     : ko_per_mb_primary (Tier 1+2, 140 KOs, kescience_mgnify)')
print(f'Min isolates  : ≥5 BacDive records per genus')
print(f'N_max (95th pctile of {_NICHE_COL}): {_N_max:.1f}')
print(f'n genera      : {res_bdive.get("n")}')
print(f'β             = {res_bdive.get("beta", float("nan")):.5f}')
print(f'SE            = {res_bdive.get("SE",   float("nan")):.5f}')
print(f't             = {res_bdive.get("t_stat", float("nan")):.3f}')
print(f'p             = {res_bdive.get("p_value", float("nan")):.3e}')
print(f'λ             = {res_bdive.get("lambda_est", float("nan")):.3f}')
print(f'r²            = {res_bdive.get("r2", float("nan")):.4f}')
print(f'Direction     : {_direction}')
if res_bdive_geo is not None:
    print(f'Secondary (geo_cat): n={res_bdive_geo.get("n")}, '
          f'β={res_bdive_geo.get("beta", float("nan")):.5f}, '
          f'p={res_bdive_geo.get("p_value", float("nan")):.3e}')
    _geo_dir = ('same direction' if res_bdive_geo.get('beta', 0) < 0 else 'opposite direction')
    print(f'  Secondary direction: {_geo_dir} as primary')
print()
print('Add to INTERPRETATION_TABLE.md §7 (Exploratory Findings):')
print(
    f'| BacDive geographic niche breadth PGLS | NB09 | nb090014 | '
    f'BacDive B_std ({_NICHE_LABEL}) ~ ko_per_mb_primary: '
    f'β={res_bdive.get("beta", float("nan")):.4f}, '
    f'p={res_bdive.get("p_value", float("nan")):.3e}, '
    f'n={res_bdive.get("n")}, λ={res_bdive.get("lambda_est", float("nan")):.3f}. '
    f'{_direction}. Niche min: ≥5 isolates, N_max={_N_max:.0f} countries (95th pctile). '
    f'| Moderate — culture-based cosmopolitanism from BacDive, independent of 16S | '
    f'Report in thesis as independent validation attempt. |'
)
